# 02 - 股票池管理

本 Notebook 完成以下任务：
1. 同步全 A 股列表到 `stock_info`
2. 同步 ETF 列表到 `etf_info`
3. 构建自选股票池 `stock_pool`（支持分组和标签）
4. 股票池增删改查演示

---

## 设计方案

### 三张表的关系
- `stock_info`：全量 A 股基础信息（约 5000+），从 Tushare 同步
- `etf_info`：全量 ETF 基础信息，从 Tushare 同步
- `stock_pool`：自选股票池，支持 `pool_group` 分组（如 "core", "watchlist", "etf"）

### 数据采集只针对 stock_pool 中的股票
日线、财务等数据只采集 `stock_pool` 中的标的，避免拉取全市场数据。

## 1. 同步 A 股列表

In [1]:
from invest_model.db import get_engine
from invest_model.sources.tushare_client import TushareClient
from invest_model.collectors.stock_list_collector import StockListCollector

engine = get_engine()
ts_client = TushareClient()
collector = StockListCollector(ts_client, engine)

stock_df = collector.collect_stock_list()
print(f"\nA 股总数: {len(stock_df)}")
print(f"\n行业分布（前10）:")
print(stock_df['industry'].value_counts().head(10).to_string())

16:04:53 | INFO    | Tushare 使用自定义接口基地址: http://118.89.66.41:8010/
16:04:55 | INFO    | Tushare 客户端初始化完成（Token 验证通过）
16:04:55 | INFO    | 开始采集 A 股股票列表...
16:04:58 | INFO    | 获取到 5499 只 A 股
16:04:59 | INFO    | 写入 stock_info: 5499 条



A 股总数: 5499

行业分布（前10）:
industry
电气设备    342
元器件     304
专用机械    285
软件服务    281
汽车配件    262
化工原料    256
半导体     193
医疗保健    182
化学制药    148
通信设备    136


## 2. 同步 ETF 列表

In [2]:
etf_df = collector.collect_etf_list()
print(f"ETF 总数: {len(etf_df)}")
if not etf_df.empty and 'fund_type' in etf_df.columns:
    print(f"\n基金类型分布:")
    print(etf_df['fund_type'].value_counts().head(10).to_string())

16:04:59 | INFO    | 开始采集 ETF 列表...
16:05:03 | INFO    | 获取到 1941 只 ETF
16:05:04 | INFO    | 写入 etf_info: 1941 条


ETF 总数: 1941

基金类型分布:
fund_type
股票型      1537
混合型       172
债券型       105
REITs      79
货币市场型      27
商品型        18
QDII        2
另类投资型       1


## 3. 构建自选股票池

将关注的个股和 ETF 加入 `stock_pool` 表。后续所有数据采集只针对池中标的。

In [3]:
from invest_model.repositories.stock_pool_repo import StockPoolRepository
from sqlalchemy import text
import pandas as pd

pool_repo = StockPoolRepository(engine)

# 清空旧的股票池，重新写入
with engine.begin() as conn:
    conn.execute(text("DELETE FROM stock_pool"))
print("已清空旧股票池\n")

# 核心个股
core_stocks = pd.DataFrame([
    {"code": "300442.SZ", "name": "润泽科技", "pool_group": "core", "tags": "IDC,数据中心"},
    {"code": "002594.SZ", "name": "比亚迪",   "pool_group": "core", "tags": "新能源车,汽车"},
    {"code": "600691.SH", "name": "潞化科技", "pool_group": "core", "tags": "煤化工,化工"},
    {"code": "002648.SZ", "name": "卫星化学", "pool_group": "core", "tags": "化工,C3"},
    {"code": "000833.SZ", "name": "粤桂股份", "pool_group": "core", "tags": "制糖,造纸"},
])
n = pool_repo.batch_add_to_pool(core_stocks)
print(f"添加核心个股: {n} 只")

# ETF
etf_pool = pd.DataFrame([
    {"code": "516120.SH", "name": "化工50ETF", "pool_group": "etf", "tags": "行业ETF,化工"},
])
n = pool_repo.batch_add_to_pool(etf_pool)
print(f"添加 ETF: {n} 只")

已清空旧股票池

添加核心个股: 5 只
添加 ETF: 1 只


## 4. 查看股票池

In [4]:
# 查看全部分组
groups = pool_repo.get_pool_groups()
print(f"股票池分组: {groups}\n")

# 查看各分组内容
for grp in groups:
    df = pool_repo.get_pool(grp)
    print(f"[{grp}] {len(df)} 只:")
    print(df[['code', 'name', 'tags']].to_string(index=False))
    print()

股票池分组: ['core', 'etf']

[core] 5 只:
     code name     tags
000833.SZ 粤桂股份    制糖,造纸
002594.SZ  比亚迪  新能源车,汽车
002648.SZ 卫星化学    化工,C3
300442.SZ 润泽科技 IDC,数据中心
600691.SH 潞化科技   煤化工,化工

[etf] 1 只:
     code    name     tags
516120.SH 化工50ETF 行业ETF,化工



In [5]:
# 获取所有代码（后续采集用）
all_codes = pool_repo.get_pool_codes()
print(f"股票池总计 {len(all_codes)} 只: {all_codes}")

股票池总计 6 只: ['000833.SZ', '002594.SZ', '002648.SZ', '300442.SZ', '600691.SH', '516120.SH']


## 完成

股票池已构建完毕。你可以随时通过 `pool_repo.add_to_pool()` / `remove_from_pool()` 调整。

继续 `03_daily_kline.ipynb` 采集日线行情数据。